# Semantic Model Similarity

Finds duplicate, near-duplicate, and subset semantic models from the metadata catalog. Each model becomes a signature (tables, columns, measures, DAX, relationships, data sources); every pair gets a symmetric **composite** score and a directional **containment** score, and duplicates are grouped into clusters. Run the notebook top to bottom to score every model pair and write the results back to the lakehouse. Open the companion **`semantic_model_similarity_results`** notebook for the interactive app.

**Needs:** a Fabric notebook with the TOM catalog tables in the attached lakehouse (`semantic_models`, `semantic_model_tables`, `semantic_model_columns`, `semantic_model_relationships`, `semantic_model_measures`, `semantic_model_datasources`). Similarity uses a local scikit-learn TF-IDF vectorizer — no external endpoint or GPU required.

## Parameters

Tier thresholds, signal weights, and write mode. The cell below is collapsed to keep the results close — click **Show input** to adjust it, then re-run.

In [ ]:
# Results are read from and written to the lakehouse attached to this notebook.
WRITE_MODE = "overwrite"  # "overwrite" replaces prior output; use "append" only if downstream supports it.

# Blocking limits comparisons to model pairs that share at least one table or measure name.
# Disable to force full pairwise comparison (slower on large catalogs).
ENABLE_BLOCKING = True

# Composite-score tier thresholds.
DUPLICATE_THRESHOLD = 0.95
SIMILAR_THRESHOLD = 0.70

# Containment threshold. Containment is a second, directional score that answers a
# different question than the composite similarity score: "does one model contain
# everything in the other?" rather than "how alike are the two models overall?".
# A pair whose stronger direction reaches this value is flagged as a containment
# candidate, and the two scores are reported side by side so you can filter on both.
CONTAINMENT_THRESHOLD = 0.95

# Report knobs.
TOP_N = 20  # Rows shown in the ranked pair table.
HEATMAP_MIN_SCORE = SIMILAR_THRESHOLD  # Hide heatmap cells scoring below this composite value.

# Relative weights for each similarity signal. Values are normalized, so they need not sum to 1.
SIMILARITY_WEIGHTS = {
    "tables": 0.15,
    "columns": 0.20,
    "measure_names": 0.15,
    "measure_dax_embedding": 0.25,
    "relationships": 0.15,
    "datasources": 0.10,
}

# Relative weights for each containment signal. Containment uses exact measure
# definitions (name + comment-stripped DAX) instead of the TF-IDF embedding, because
# lexical cosine similarity is symmetric and cannot establish that one model's measures
# are a subset of another's. Weights are normalized per direction over whichever signals
# the source model actually has.
CONTAINMENT_WEIGHTS = {
    "tables": 0.15,
    "columns": 0.20,
    "measure_names": 0.15,
    "measure_definitions": 0.25,
    "relationships": 0.15,
    "datasources": 0.10,
}

# Guard against misconfigured weights before any scoring runs.
for _weights_name, _weights in (
    ("SIMILARITY_WEIGHTS", SIMILARITY_WEIGHTS),
    ("CONTAINMENT_WEIGHTS", CONTAINMENT_WEIGHTS),
):
    if any(weight < 0 for weight in _weights.values()):
        raise ValueError(f"{_weights_name} must not contain negative weights.")
    if sum(_weights.values()) <= 0:
        raise ValueError(f"{_weights_name} must sum to a positive value.")


## Prepare the data

Loads the catalog and scores every model pair. All inputs here are hidden — expand any cell to inspect the logic; you don't need to change anything.

In [ ]:
import itertools
import re
from collections import defaultdict

import numpy as np
import pandas as pd

In [ ]:
def load_delta(table_name):
    df = spark.read.format("delta").load('Tables/' + table_name)
    return df.toPandas()


models_df = load_delta("semantic_models")
tables_df = load_delta("semantic_model_tables")
columns_df = load_delta("semantic_model_columns")
relationships_df = load_delta("semantic_model_relationships")
measures_df = load_delta("semantic_model_measures")
datasources_df = load_delta("semantic_model_datasources")

if models_df.empty:
    raise ValueError(
        "No rows in the semantic_models table of the attached lakehouse. "
        "Run the TOM catalog notebook first."
    )

print(f"Models: {len(models_df)}")
print(
    f"Tables: {len(tables_df)} | Columns: {len(columns_df)} | "
    f"Measures: {len(measures_df)} | Relationships: {len(relationships_df)} | "
    f"Datasources: {len(datasources_df)}"
)

In [ ]:
def norm(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip().casefold()


def norm_dax(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    text = str(value)
    text = re.sub(r"/\*.*?\*/", " ", text, flags=re.S)  # block comments
    text = re.sub(r"//.*", " ", text)  # line comments
    text = re.sub(r"\s+", " ", text)
    return text.strip().casefold()


def model_label(sig):
    return f"{sig['workspace_name']} / {sig['model_name']}"


signatures = {}
for _, row in models_df.iterrows():
    model_id = str(row["model_id"])
    signatures[model_id] = {
        "model_id": model_id,
        "workspace_id": str(row.get("workspace_id", "")),
        "workspace_name": str(row.get("workspace_name", "")),
        "model_name": str(row.get("model_name", "")),
        "tables": set(),
        "columns": set(),
        "measure_names": set(),
        "measure_definitions": set(),
        "relationships": set(),
        "datasources": set(),
        "dax_docs": [],
    }


def sig_for(model_id):
    return signatures.get(str(model_id))


for _, row in tables_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        sig["tables"].add(norm(row["table_name"]))

for _, row in columns_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        sig["columns"].add(f"{norm(row['table_name'])}.{norm(row['column_name'])}")

for _, row in measures_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        measure_name = norm(row["measure_name"])
        measure_dax = norm_dax(row.get("expression"))
        sig["measure_names"].add(measure_name)
        # Name + DAX key so containment only credits measures whose logic also matches.
        sig["measure_definitions"].add(f"{measure_name} :: {measure_dax}")
        sig["dax_docs"].append(f"{measure_name} {measure_dax}".strip())

for _, row in relationships_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        key = (
            f"{norm(row['from_table'])}.{norm(row['from_column'])}"
            f"->{norm(row['to_table'])}.{norm(row['to_column'])}"
        )
        sig["relationships"].add(key)

for _, row in datasources_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        conn = row.get("connection_string") or row.get("connection_details") or row.get("datasource_name")
        conn_norm = norm(conn)
        if conn_norm:
            sig["datasources"].add(conn_norm)

# Build the embedding document per model, with a structural fallback when no measures exist.
for sig in signatures.values():
    parts = list(sig["dax_docs"])
    if not parts:
        parts = sorted(sig["tables"]) + sorted(sig["columns"])
    sig["doc"] = " \n ".join(parts) if parts else (sig["model_name"] or sig["model_id"])

model_ids = list(signatures.keys())
print(f"Built signatures for {len(model_ids)} models.")


In [ ]:
def jaccard(set_a, set_b):
    if not set_a and not set_b:
        return 0.0
    union = len(set_a | set_b)
    return len(set_a & set_b) / union if union else 0.0


def coverage(source_set, other_set):
    # Directional: the fraction of source_set's members that also appear in other_set.
    # Returns None when the signal is absent from the source, so it can be excluded from
    # the weighted average rather than counted as a spurious perfect match.
    if not source_set:
        return None
    return len(source_set & other_set) / len(source_set)


def weighted_containment(source_sig, other_sig, weights):
    # How completely source_sig is contained in other_sig: a weighted mean of the
    # per-signal coverages, normalized over whichever signals the source actually has.
    accumulated = 0.0
    total_weight = 0.0
    for signal, weight in weights.items():
        signal_coverage = coverage(source_sig[signal], other_sig[signal])
        if signal_coverage is None:
            continue
        accumulated += weight * signal_coverage
        total_weight += weight
    return accumulated / total_weight if total_weight else 0.0


def classify_containment(a_in_b, b_in_a, threshold):
    a_contained = a_in_b >= threshold
    b_contained = b_in_a >= threshold
    if a_contained and b_contained:
        return "equivalent"
    if b_contained:
        return "model_a_contains_model_b"
    if a_contained:
        return "model_b_contains_model_a"
    return "partial_overlap"


if ENABLE_BLOCKING:
    block_index = defaultdict(set)
    for model_id, sig in signatures.items():
        for table_name in sig["tables"]:
            block_index[("t", table_name)].add(model_id)
        for measure_name in sig["measure_names"]:
            block_index[("m", measure_name)].add(model_id)
    candidate_pairs = set()
    for group in block_index.values():
        if len(group) > 1:
            candidate_pairs.update(itertools.combinations(sorted(group), 2))
else:
    candidate_pairs = set(itertools.combinations(sorted(model_ids), 2))

print(f"Candidate pairs to score: {len(candidate_pairs)}")


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF avoids the torch/transformers dependency chain (the Fabric runtime ships
# an older PyTorch than recent transformers require). Rows are L2-normalized, so
# cosine similarity stays a plain dot product for the downstream scoring step.
docs = [signatures[model_id]["doc"] for model_id in model_ids]
vectorizer = TfidfVectorizer(min_df=1, norm="l2")

embeddings = vectorizer.fit_transform(docs).toarray()
print(f"Encoded {len(docs)} model documents into {embeddings.shape[1]}-dim TF-IDF vectors.")
embedding_index = {model_id: idx for idx, model_id in enumerate(model_ids)}

In [ ]:
weight_sum = sum(SIMILARITY_WEIGHTS.values())
pair_rows = []

for model_id_a, model_id_b in candidate_pairs:
    sig_a = signatures[model_id_a]
    sig_b = signatures[model_id_b]

    j_tables = jaccard(sig_a["tables"], sig_b["tables"])
    j_columns = jaccard(sig_a["columns"], sig_b["columns"])
    j_measures = jaccard(sig_a["measure_names"], sig_b["measure_names"])
    j_relationships = jaccard(sig_a["relationships"], sig_b["relationships"])
    j_datasources = jaccard(sig_a["datasources"], sig_b["datasources"])
    cosine = float(
        np.dot(embeddings[embedding_index[model_id_a]], embeddings[embedding_index[model_id_b]])
    )
    cosine = max(0.0, min(1.0, cosine))

    composite = (
        SIMILARITY_WEIGHTS["tables"] * j_tables
        + SIMILARITY_WEIGHTS["columns"] * j_columns
        + SIMILARITY_WEIGHTS["measure_names"] * j_measures
        + SIMILARITY_WEIGHTS["measure_dax_embedding"] * cosine
        + SIMILARITY_WEIGHTS["relationships"] * j_relationships
        + SIMILARITY_WEIGHTS["datasources"] * j_datasources
    ) / weight_sum

    # Directional containment: how much of each model is absorbed by the other.
    a_in_b = weighted_containment(sig_a, sig_b, CONTAINMENT_WEIGHTS)
    b_in_a = weighted_containment(sig_b, sig_a, CONTAINMENT_WEIGHTS)
    containment_score = max(a_in_b, b_in_a)
    containment_relationship = classify_containment(a_in_b, b_in_a, CONTAINMENT_THRESHOLD)

    if composite >= DUPLICATE_THRESHOLD:
        tier = "duplicate"
    elif composite >= SIMILAR_THRESHOLD:
        tier = "similar"
    else:
        tier = "distinct"

    pair_rows.append({
        "model_id_a": model_id_a,
        "model_a": model_label(sig_a),
        "workspace_a": sig_a["workspace_name"],
        "model_id_b": model_id_b,
        "model_b": model_label(sig_b),
        "workspace_b": sig_b["workspace_name"],
        "same_model_name": norm(sig_a["model_name"]) == norm(sig_b["model_name"]),
        "cross_workspace": sig_a["workspace_id"] != sig_b["workspace_id"],
        "jaccard_tables": round(j_tables, 4),
        "jaccard_columns": round(j_columns, 4),
        "jaccard_measure_names": round(j_measures, 4),
        "jaccard_relationships": round(j_relationships, 4),
        "jaccard_datasources": round(j_datasources, 4),
        "dax_embedding_cosine": round(cosine, 4),
        "composite_score": round(composite, 4),
        "containment_score": round(containment_score, 4),
        "containment_relationship": containment_relationship,
        "model_a_in_model_b": round(a_in_b, 4),
        "model_b_in_model_a": round(b_in_a, 4),
        "tier": tier,
    })

pairs_df = pd.DataFrame(pair_rows)
if not pairs_df.empty:
    pairs_df = pairs_df.sort_values("composite_score", ascending=False).reset_index(drop=True)

# Union-find clustering over duplicate-tier pairs.
parent = {model_id: model_id for model_id in model_ids}


def find(node):
    while parent[node] != node:
        parent[node] = parent[parent[node]]
        node = parent[node]
    return node


def union(node_a, node_b):
    root_a, root_b = find(node_a), find(node_b)
    if root_a != root_b:
        parent[root_a] = root_b


if not pairs_df.empty:
    for _, row in pairs_df[pairs_df["tier"] == "duplicate"].iterrows():
        union(row["model_id_a"], row["model_id_b"])

cluster_members = defaultdict(list)
for model_id in model_ids:
    cluster_members[find(model_id)].append(model_id)

cluster_rows = []
cluster_number = 0
for members in cluster_members.values():
    if len(members) > 1:
        cluster_number += 1
        for model_id in members:
            sig = signatures[model_id]
            cluster_rows.append({
                "cluster_id": cluster_number,
                "cluster_size": len(members),
                "model_id": model_id,
                "model": model_label(sig),
                "workspace_name": sig["workspace_name"],
                "model_name": sig["model_name"],
            })

clusters_df = pd.DataFrame(cluster_rows)

# Per-model signature summary.
signature_rows = []
for sig in signatures.values():
    signature_rows.append({
        "model_id": sig["model_id"],
        "workspace_name": sig["workspace_name"],
        "model_name": sig["model_name"],
        "table_count": len(sig["tables"]),
        "column_count": len(sig["columns"]),
        "measure_count": len(sig["measure_names"]),
        "relationship_count": len(sig["relationships"]),
        "datasource_count": len(sig["datasources"]),
    })
signatures_df = pd.DataFrame(signature_rows)

duplicate_count = int((pairs_df["tier"] == "duplicate").sum()) if not pairs_df.empty else 0
similar_count = int((pairs_df["tier"] == "similar").sum()) if not pairs_df.empty else 0
containment_count = (
    int((pairs_df["containment_score"] >= CONTAINMENT_THRESHOLD).sum()) if not pairs_df.empty else 0
)
print(f"Duplicate pairs: {duplicate_count} | Similar pairs: {similar_count} | Containment pairs: {containment_count}")
print(f"Duplicate clusters: {cluster_number}")


## Save results to the lakehouse

Write the per-model signature summary, the full pairwise scores, and the duplicate clusters as Delta tables so the analysis is queryable outside this notebook.

In [ ]:
similarity_outputs = {
    "semantic_model_signatures": signatures_df,
    "semantic_model_similarity_pairs": pairs_df,
    "semantic_model_duplicate_clusters": clusters_df,
}

for table_name, frame in similarity_outputs.items():
    if frame.empty:
        print(f"Skipped {table_name}: no rows")
        continue
    spark.createDataFrame(frame).write.format("delta").mode(WRITE_MODE).option(
        "overwriteSchema", "true"
    ).saveAsTable(table_name)
    print(f"Wrote {len(frame)} rows to {table_name}")

# Run metadata for the companion Results notebook: default thresholds + summary counts.
from datetime import datetime, timezone

_run_meta = pd.DataFrame([{
    "generated_at": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
    "duplicate_threshold": float(DUPLICATE_THRESHOLD),
    "similar_threshold": float(SIMILAR_THRESHOLD),
    "containment_threshold": float(CONTAINMENT_THRESHOLD),
    "model_count": int(len(signatures_df)) if not signatures_df.empty else int(len(signatures)),
    "pair_count": int(len(pairs_df)),
    "duplicate_count": int(duplicate_count),
    "similar_count": int(similar_count),
    "containment_count": int(containment_count),
    "cluster_count": int(cluster_number),
}])
spark.createDataFrame(_run_meta).write.format("delta").mode(WRITE_MODE).option(
    "overwriteSchema", "true"
).saveAsTable("semantic_model_similarity_run")
print("Wrote run metadata to semantic_model_similarity_run")

## Next steps

- In the **`semantic_model_similarity_results`** notebook, start with the **Overview** and **Clusters** tabs — the strongest consolidation candidates. A `cross-workspace` tag often means the same model was copied between workspaces.
- Open the **Containment** tab to find subset relationships: one model holding everything in another (plus more) is a superseding model or an extract that may be retired. `semantic_model_similarity_pairs` carries both `composite_score` and `containment_score`, so you can filter on either independently (for example, high containment with only moderate similarity = a small model absorbed by a much larger one).
- Similarity and containment are metadata-based. Before acting on a pair, confirm the models truly serve the same purpose; they do not compare report layouts, row-level data, refresh history, or security roles.
- If the tiers look too strict or too loose, calibrate `DUPLICATE_THRESHOLD`, `SIMILAR_THRESHOLD`, `CONTAINMENT_THRESHOLD`, `SIMILARITY_WEIGHTS`, and `CONTAINMENT_WEIGHTS` in the Parameters cell against a few known pairs, then re-run.
- The output tables (`semantic_model_signatures`, `semantic_model_similarity_pairs`, `semantic_model_duplicate_clusters`, and `semantic_model_similarity_run`) are written to the attached lakehouse. The companion **`semantic_model_similarity_results`** notebook reads them to render the interactive app, so you never need to edit code to explore the results.